# Paleomagnetism

The `apsg.paleomag` module covers the standard workflow of a paleomagnetic study: it reads demagnetization data of oriented drill-core specimens, isolates the remanence components by principal component analysis, averages the specimens of a site and provides the tests used to decide when and how the rock acquired its magnetization - fold and reversal tests - as well as virtual geomagnetic poles and inclination-only estimates. It is not imported by `from apsg import *`, so we import it separately.

The workflow of this tutorial follows the steps of a real study:

1. **Specimens**: read the demagnetization steps, look at them in specimen, geographic and tilt-corrected coordinates.
2. **Components**: fit lines (and great circles) to selected steps by principal component analysis (Kirschvink, 1980).
3. **Site means**: Fisher statistics of the components of the specimens of a site, combined means of lines and great circles.
4. **Age of the magnetization**: fold test (before or after tilting?) and reversal test (do both polarities share an axis?).
5. **Poles and paleolatitude**, and the mean inclination when only inclinations are known.

In [ ]:
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from apsg import *
from apsg.paleomag import *

## The data

The examples use an **artificial** data set: 71 drill-core specimens of 10 sites (for example lava flows or beds), made by the script `scripts/make_pmdata.py` of the repository with a fixed random seed. Names, location and numbers are invented, but the data are generated the way a real experiment behaves, so they look like measured data:

- every site has its own bedding and a characteristic remanence (ChRM) of magnetite-like unblocking temperatures, acquired **before** the tilting; it is of normal polarity at seven sites and reversed at three;
- a soft overprint along the present field is removed by heating to about 300 °C;
- a few specimens have overlapping components and follow great circles, a few are noisy or were not heated to the end of the schedule;
- the cores have random orientation and the magnetization is measured with an instrument noise floor.

The site table holds the (hypothetical) location and the bedding of each site.

In [ ]:
sites = pd.read_csv('pmdata/sites.csv', index_col='site')
sites

## Specimens

A `Specimen` holds the demagnetization steps of one core with its orientation (core azimuth and hade) and the bedding. `Specimen.from_file` reads PMD, Remasoft RS3 and SQUID files (`from_pmd`, `from_rs3`, `from_squid`), the format is recognized by the extension. The magnetization is stored in specimen coordinates (A/m) and rotated on demand into **geographic** coordinates (in situ) and **tilt-corrected** coordinates (bedding restored to horizontal).

In [ ]:
spec = Specimen.from_pmd('pmdata/PM01.1a.pmd')
spec

The steps are available as arrays or as a table with the directions in all three coordinate systems: specimen (`sp`), geographic (`ge`) and tilt corrected (`tc`).

In [ ]:
spec.to_dataframe().head(8).round(1)

A specimen is usually inspected by two plots. The decay plot shows the remaining intensity against the treatment, and the Zijderveld (orthogonal projection) plot shows the endpoints of the magnetization vector in two projections, horizontal (blue) and vertical (green). A component that decays to the origin is a **straight segment** pointing to it. Here the soft overprint is removed by about 250-300 °C, which makes the first steps curved, and the characteristic component is the straight line toward the origin at higher temperatures.

In [ ]:
fig, (a, b, c) = plt.subplots(1, 3, figsize=(15, 5))
demag_plot(spec, ax=a)
zijderveld_plot(spec, coords='geo', ax=b)
zijderveld_plot(spec, coords='tilt', ax=c)
plt.tight_layout()

## Components

Components are fitted to a selection of steps by principal component analysis (Kirschvink, 1980). Steps are selected by slice, position, or by treatment level with `between(lo, hi)`. `interpret` fits and stores the component under a name (here `"MAG"`) and returns it in geographic coordinates, `pca` only returns the fit.

`Component` holds the direction `vector`, the maximum angular deviation `mad` (the scatter of the steps about the line; below about 5-10° is a well-defined direction) and the deviation angle `dang` (the angle between the line and the line from the origin to the centre of the steps; a small `dang` shows that the line points to the origin, i.e. that the component is trustworthy).

In [ ]:
steps = spec.between(400, 580)
mag = spec.interpret('MAG', steps.labels)
mag

In [ ]:
print(f'dec {mag.vector.geo[0]:.1f} inc {mag.vector.geo[1]:.1f}  MAD {mag.mad:.1f}  DANG {mag.dang:.1f}  ({len(steps)} steps)')

Components are stored in specimen coordinates and rotated on demand, so the same component is available in geographic or tilt-corrected coordinates. The stereonet plot shows the steps in the chosen coordinates with the fitted component.

In [ ]:
spec.component('MAG', coords='tilt')

In [ ]:
stereo_plot(spec, coords='geo', show=False).show()

An anchored fit is forced through the origin, which is appropriate for a component that decays to the origin (it is then described by one point less). A `kind="plane"` fit returns a great circle, described by its pole (see below).

In [ ]:
spec.pca(steps.labels, anchor=True)

## Site means

A `Sample` collects specimens, for example the specimens of one site. The function below reads all specimens of a site, sets the location from the site table and fits the MAG component over the steps between 400 and 580 °C, the range where the overprint has vanished and the characteristic component is not yet lost in the noise. (In a real study the range is chosen for every specimen after looking at its Zijderveld plot.)

In [ ]:
def read_site(site, lo=400, hi=580):
    smp = Sample.from_files(sorted(glob(f'pmdata/{site}.*.pmd')), name=site)
    for s in smp:
        s.latitude, s.longitude = sites.latitude[site], sites.longitude[site]
        s.interpret('MAG', s.between(lo, hi).labels)
    return smp


samples = {site: read_site(site) for site in sites.index}
samples['PM01']

`mean` gives the Fisher statistics of the components in any coordinate system (`"specimen"`, `"geo"` or `"tilt"`): mean direction `mu`, precision `k` and the radius `alpha` of the 95% confidence cone.

In [ ]:
samples['PM01'].mean('MAG', coords='geo')

In [ ]:
samples['PM01'].mean('MAG', coords='tilt')

The precision does not change with the tilt correction of a single site, as it is a rotation of all its specimens. The table shows the means of all sites in both coordinate systems. In geographic coordinates the directions are scattered because the beds are tilted differently, after the tilt correction they fall (apart from the polarity) together.

In [ ]:
rows = []
for site, smp in samples.items():
    row = {'site': site, 'n': len(smp), 'bed': f'{sites.bed_dip_direction[site]}/{sites.bed_dip[site]}'}
    for coords in ('geo', 'tilt'):
        m = smp.mean('MAG', coords)
        dec, inc = m['mu'].geo
        row.update({f'dec_{coords}': dec, f'inc_{coords}': inc})
    row.update({'k': m['k'], 'alpha95': m['alpha']})
    rows.append(row)
pd.DataFrame(rows).set_index('site').round(1)

The directions of all specimens in geographic and in tilt-corrected coordinates, with the specimens of a site sharing the same bedding. Filled symbols point down, open ones up (reversed polarity).

In [ ]:
everything = Sample([s for smp in samples.values() for s in smp], name='All sites')
s = StereoNet(title='Geographic')
s.vector(everything.directions('MAG', 'geo'), color='C0', alpha=0.6)
s.show()

In [ ]:
s = StereoNet(title='Tilt corrected')
s.vector(everything.directions('MAG', 'tilt'), color='C1', alpha=0.6)
s.show()

## Polarity

After the tilt correction the directions form two antipodal groups, normal (down, northerly) and reversed (up, southerly). A Fisher mean of all specimens would be meaningless because the two polarities cancel, so the sample is first divided by the polarity (here by the sign of the inclination in tilt-corrected coordinates). The Fisher statistics then show how the tilt correction improves the clustering of each polarity group, which is the idea of the fold test.

In [ ]:
def polarity(spec):
    return 'N' if spec.component('MAG', 'tilt').vector.geo[1] > 0 else 'R'


normal = Sample([s for s in everything if polarity(s) == 'N'], name='Normal')
reversed_ = Sample([s for s in everything if polarity(s) == 'R'], name='Reversed')
print(len(normal), 'normal and', len(reversed_), 'reversed specimens')
for smp in (normal, reversed_):
    for coords in ('geo', 'tilt'):
        m = smp.mean('MAG', coords)
        dec, inc = m['mu'].geo
        print(f"{smp.name:9s} {coords:5s} dec {dec:5.1f} inc {inc:5.1f}  k {m['k']:6.1f}  alpha95 {m['alpha']:.1f}")

## Great circles and combined means

When a component cannot be isolated, the demagnetization path of a specimen follows a great circle toward the sought direction, which gives a constraint instead of a point. A `kind="plane"` fit returns the great circle, described by its pole. In the site PM04 the paths of three specimens curve (the overprint and the characteristic component unblock in overlapping ranges). We replace their line fits by great circles fitted to the steps from 150 to 560 °C. The MAD of the circle tells how well it is defined, and the line fits lie close to the circles.

In [ ]:
pm04 = read_site('PM04')
for name in ['PM04.2a', 'PM04.5a', 'PM04.7a']:
    s = pm04[name]
    line = s.component('MAG', 'geo')
    circle = s.interpret('MAG', s.between(150, 560).labels, kind='plane')
    dist = abs(90 - circle.vector.angle(line.vector))
    print(f'{name}: MAD of the circle {circle.mad:.1f}, line {dist:.1f} degrees from the circle')

`Sample.mean` combines the directions and the great circles by the method of McFadden and McElhinny (1988): each circle contributes its point closest to the mean, iterated to convergence, and the precision and the confidence cone account for the weaker constraint of the circles. We compare it with the Fisher mean of the line fits of all specimens.

In [ ]:
combined = pm04.mean('MAG', 'geo')
lines_only = read_site('PM04').mean('MAG', 'geo')
for label, m in (('lines only', lines_only), ('lines and circles', combined)):
    dec, inc = m['mu'].geo
    print(f"{label:18s} dec {dec:.1f} inc {inc:.1f} alpha95 {m['alpha']:.1f} ({m['n_lines']} lines, {m['n_planes']} circles)")

The underlying functions work on arrays of directions and poles. `kirschvink_mean` is the least-squares method of Kirschvink (1980), which has no confidence region, and `bailey_halls_mean` is the maximum likelihood solution of Bailey and Halls (1984), which gives a confidence ellipse. Great circles alone give a biased mean when they come from two populations (Schmidt, 1985), so direct observations of the sought direction should be included.

In [ ]:
comps = [s.component('MAG', 'geo') for s in pm04]
lines = [c.vector for c in comps if c.kind == 'line']
poles = [c.vector for c in comps if c.kind == 'plane']
kirschvink_mean(lines, poles)['mu'].geo

In [ ]:
bh = bailey_halls_mean(lines, poles)
print(bh['mu'].geo, f"k {bh['k']:.0f}, ellipse semi-axes {bh['semi_axes'][0]:.1f} x {bh['semi_axes'][1]:.1f}")

## Fold tests

A magnetization older than the folding (here: tilting) has the same direction in all sites after the tilt correction, while a younger one (acquired after tilting) is the same *before* it. The sites of the tutorial have different dip directions and dips between 18 and 60 degrees, which gives the test its power.

The bootstrap fold test of Tauxe and Watson (1994) finds the percentage of untilting at which the largest eigenvalue $\tau_1$ of the orientation tensor of the directions is maximal (the directions are the most clustered), for many resamples of the data. The orientation tensor does not depend on the sign of the vectors, so both polarities can be tested together. A confidence interval of the maxima that includes 100% supports a pre-folding magnetization, an interval around 0% a post-folding one.

In [ ]:
ft = everything.fold_test('MAG', n=1000, rng=0)
print(f"best untilting {ft['best']:.0f} %, 95 % interval {ft['ci'][0]:.0f} to {ft['ci'][1]:.0f} %")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(ft['fracs'], ft['tau1'])
ax.axvspan(*ft['ci'], alpha=0.2, label='95 % interval of the maxima')
ax.set_xlabel('Untilting [%]')
ax.set_ylabel(r'$\tau_1$')
ax.legend()

The directions are best clustered at about complete untilting and the interval includes 100 %, so the magnetization is older than the tilting - which is how the data were constructed.

The test of McFadden (1990) works with the **mean directions of the sites**, each one on its own limb of a fold. For a chosen percentage of untilting it calculates the correlation `xi` between the deviations of the directions from their mean and the displacement of the mean by a small tilt about the strike of each bed. Directions that scatter because they are tilted differently correlate with the tilt, so a large `|xi|` shows that they are not in their primary position. The statistic is compared with a critical value of the null distribution (Lord, 1948), and the percentage of untilting with the smallest `|xi|` is the best estimate. A magnetization acquired before the folding is significant in situ but not when unfolded. The test needs a single polarity, so the means of the reversed sites are turned to their antipodes.

In [ ]:
sign = {site: 1 if smp.mean('MAG', 'tilt')['mu'].geo[1] > 0 else -1 for site, smp in samples.items()}
directions = [sign[site] * np.asarray(smp.mean('MAG', 'geo')['mu']) for site, smp in samples.items()]
beddings = [np.asarray(smp[0].bedding) for smp in samples.values()]
mf = mcfadden_fold_test(directions, beddings)
print(f"xi in situ {mf['geo']:.2f}, unfolded {mf['tilt']:.2f}, critical value {mf['critical']:.2f}, N = {mf['n']}")
print(f"smallest |xi| at {mf['best']:.0f} % untilting")
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(mf['fracs'], mf['xi'])
ax.axhline(mf['critical'], color='r', ls='--', lw=1)
ax.axhline(-mf['critical'], color='r', ls='--', lw=1, label='critical value')
ax.set_xlabel('Untilting [%]')
ax.set_ylabel(r'$\xi$')
ax.legend()

The in situ statistic exceeds the critical value, so the directions are significantly correlated with the tilts, while the unfolded position is far from significant and the smallest |xi| is at complete untilting: the magnetization was acquired before the tilting, in agreement with the bootstrap test. The same test can be run on the specimens of a `Sample` with `Sample.mcfadden_fold_test`, treating every specimen as a site.

## Reversal test

A paleomagnetic direction averaged over enough time is symmetric: the reversed polarity direction is antipodal to the normal one. The **reversal test** asks whether the two groups, after turning the reversed one to its antipode, share a common mean. `Sample.reversal_test` splits the directions by the sign of their projection on the principal axis of the orientation tensor and by default applies Watson's test of a common mean direction. With `method="bootstrap"` it compares the bootstrap confidence intervals of the three Cartesian components of the two means, which does not assume a Fisher distribution (Tauxe, 2003). The two criteria can disagree. A result that is **not** rejected - the means are indistinguishable from antipodal - is the sought one.

In [ ]:
print(everything.reversal_test('MAG', coords='tilt'))
bt = everything.reversal_test('MAG', coords='tilt', method='bootstrap', rng=0)
print(bt, ' components with disjoint intervals:', int(bt.statistic))
print('normal  :', normal.mean('MAG', 'tilt')['mu'].geo)
print('reversed:', vec(-np.asarray(reversed_.mean('MAG', 'tilt')['mu'])).geo, ' (antipode)')

## Virtual geomagnetic pole

`vgp` calculates the virtual geomagnetic pole - the position of the north pole of the geocentric dipole that would produce the observed direction at the site - with its oval of confidence `dp` and `dm` and the paleolatitude of the site from the inclination ($\tan I = 2\tan\lambda$). `Sample.vgp` does the same for the mean direction of a sample. Because a pole of a reversed site lies at the opposite side of the globe, poles are calculated for a sample of one polarity, here the normal one, or per site.

In [ ]:
normal.vgp('MAG', coords='tilt', latitude=sites.latitude.mean(), longitude=sites.longitude.mean())

The poles of the individual sites show the scatter of the site means (secular variation). The reversed sites give poles in the opposite hemisphere, the antipodes of which agree with those of the normal sites.

In [ ]:
rows = []
for site, smp in samples.items():
    m = smp.mean('MAG', 'tilt')
    dec, inc = m['mu'].geo
    p = vgp(dec, inc, sites.latitude[site], sites.longitude[site], m['alpha'])
    rows.append({'site': site, 'polarity': 'N' if inc > 0 else 'R', 'pole_lat': p['lat'], 'pole_lon': p['lon'],
                 'dp': p['dp'], 'dm': p['dm'], 'paleolatitude': p['paleolatitude']})
pd.DataFrame(rows).set_index('site').round(1)

## Inclination-only data

Sometimes only inclinations are known, for example for unoriented drill cores. `inclination_only_mean` gives the mean inclination of Fisher distributed directions with unknown declination, the precision and the error angle (McFadden and Reid, 1982). We pretend that the declinations of the tilt-corrected MAG directions (taken to the lower hemisphere) are lost and compare the estimate with the full directions.

In [ ]:
v = np.array([np.asarray(c.vector) * np.sign(c.vector.z)
              for c in (s.component('MAG', 'tilt') for s in everything if 'MAG' in s.components)])
inc = np.degrees(np.arcsin(v[:, 2] / np.linalg.norm(v, axis=1)))
full = vecset([vec(*x) for x in v]).fisher_statistics()
only = inclination_only_mean(inc)
print(f"full directions: inc {full['mu'].geo[1]:.1f} alpha95 {full['alpha']:.1f}")
print(f"inclination only: inc {only['inc']:.1f} alpha {only['alpha']:.1f} k {only['k']:.0f}")

## Saving specimens

`Specimen` and `Sample` have `to_json` for a lossless copy, and a specimen can be written to the PMD and RS3 formats by `to_pmd` and `to_rs3`. The API of the module is listed in the [reference](../api/paleomag.md).